# Docs 5 — Fields from an LLM

For the documents where every author formatted things their own way. The
schema is the contract; validation stays mandatory, because an LLM can
invent.

**No class API key?** Every model response below is also shown precomputed
in the text, so the whole pattern is learnable without one.

In [ ]:
%pip install -q anthropic
import os, json, anthropic
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('Class API key: '))
MODEL = 'claude-opus-5'
client = anthropic.Anthropic()

def ask(prompt, max_tokens=400):
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}])
    return ''.join(b.text for b in r.content if b.type == 'text')

## The schema prompt

Same keys for every document, `null` allowed, JSON only — so every answer
lands in the same table as the rules-extracted rows.

In [ ]:
SCHEMA_PROMPT = """Extract from the document below as JSON with EXACTLY these keys:
  date (YYYY-MM-DD or null),
  families (integer or null),
  donations (number or null),
  donations_approximate (true if the document gives a rough figure, else false),
  contact (string or null).
Use null for anything the document does not state. Do not guess.
Reply with ONLY the JSON object.

DOCUMENT:
"""

june_note = """Quick note instead of the form this month, sorry! We had a
great June - somewhere around fifteen hundred dollars came in between the
two drives, and I counted 188 families across the month. - Rosa"""

raw = ask(SCHEMA_PROMPT + june_note)
print(raw)
# Precomputed (a typical run):
# {"date": null, "families": 188, "donations": 1500,
#  "donations_approximate": true, "contact": null}

Read that row against the note. `donations_approximate: true` is the
schema doing ethical work: "around fifteen hundred" recorded as a flagged
estimate, not fake precision. And `date: null` is correct — Rosa never
states one. A schema without permission to say null forces inventions.

## The retry pattern (from Build with LLMs, and it never changes)

In [ ]:
def get_json(prompt, tries=3):
    for attempt in range(tries):
        raw = ask(prompt + ' Reply with ONLY valid JSON.')
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f'attempt {attempt+1}: not valid JSON, retrying')
    return None

row = get_json(SCHEMA_PROMPT + june_note)
print(row)

## The invention hunt

Five varied documents. Extract all five, then audit every field: **stated**
(you can point at the words), **inferred** (reasonable reading), or
**invented** (not in the document). At least one invention typically hides
in a batch like this — the classic is a plausible date the document never
gives.

In [ ]:
varied = [
 "March was strong - two hundred twelve families, and donations came to $1,847.50. Reach us at pantry@example.org.",
 "Slow month. Maybe 90 families? The freezer broke Tuesday and we lost some stock.",
 "Year-end letter: across December we welcomed 251 families and received $3,010.75 in gifts.",
 "Fwd: Fwd: totals - april 11 numbers were 198 fams / $2210 even. rosa has the binder.",
 "The board thanks everyone for a wonderful spring drive. Full figures to follow next month.",
]

rows = []
for i, doc in enumerate(varied):
    row = get_json(SCHEMA_PROMPT + doc)
    rows.append(row)
    print(i + 1, row)

In [ ]:
audit = """
Doc 1 - date: ___  families: ___  donations: ___  (stated / inferred / invented)
Doc 2 -
Doc 3 -
Doc 4 -   <- careful: what year did the model give April 11, and where did it come from?
Doc 5 -   <- the honest row here is almost all null. Is it?
"""
print(audit)

## Turn-in

The five rows with your stated / inferred / invented marks per field, and
the invention you caught, quoted against its document.